### Step 1: Bronze duplicate check

In [0]:
SET spark.databricks.execution.timeout = 600;

In [0]:
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_id) AS unique_events
FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze;

In [0]:
SELECT event_id, COUNT(*) AS copies
FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze
GROUP BY event_id
HAVING COUNT(*) > 1;

### Step 2: Idempotent MERGE into silver

In [0]:
CREATE TABLE IF NOT EXISTS dbr_dev_ua5816bd.natalkamartinuk55_silver.zerobus_youtube_silver (
  event_id STRING NOT NULL,
  video_id STRING,
  user_id STRING,
  category STRING,
  watch_duration_sec INT,
  event_timestamp TIMESTAMP,
  _silver_processed_at TIMESTAMP
)
COMMENT 'Deduplicated YouTube watch events from Zerobus bronze (Lab 11)';

In [0]:
MERGE INTO dbr_dev_ua5816bd.natalkamartinuk55_silver.zerobus_youtube_silver AS t
USING (
  SELECT
    event_id,
    video_id,
    user_id,
    category,
    watch_duration_sec,
    CAST(event_timestamp AS TIMESTAMP) AS event_timestamp
  FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze
  WHERE event_id IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY event_id ORDER BY event_timestamp) = 1
) AS s
ON t.event_id = s.event_id
WHEN NOT MATCHED THEN
  INSERT (event_id, video_id, user_id, category, watch_duration_sec, event_timestamp, _silver_processed_at)
  VALUES (s.event_id, s.video_id, s.user_id, s.category, s.watch_duration_sec, s.event_timestamp, current_timestamp());

In [0]:
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT event_id) AS unique_events
FROM dbr_dev_ua5816bd.natalkamartinuk55_silver.zerobus_youtube_silver;

In [0]:
INSERT INTO dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze
SELECT * FROM dbr_dev_ua5816bd.natalkamartinuk55_bronze.zerobus_youtube_bronze
WHERE video_id = 'VID_1010'
LIMIT 1;